In [3]:
import re
import os
import joblib
import numpy as np
import pandas as pd

from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [4]:
df = pd.read_csv("data/malicious_phish.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df["type"].value_counts())

Dataset shape: (651191, 2)
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


In [ ]:
import re
import ipaddress
from urllib.parse import urlparse

suspicious_words = [
    "login", "verify", "account", "update",
    "secure", "banking", "password", "signin",
    "confirm", "wallet"
]

def extract_features(url):
    url = str(url).strip()
    url_lower = url.lower()

    # Safely parse URLs, including malformed ones
    candidate = url if "://" in url else "http://" + url

    try:
        parsed = urlparse(candidate)
        hostname = (parsed.hostname or "").lower()
        path = (parsed.path or "").lower()
        scheme = parsed.scheme.lower()
    except (ValueError, TypeError):
        hostname = ""
        path = ""
        scheme = ""

    # Detect IP addresses safely
    try:
        ipaddress.ip_address(hostname)
        has_ip = 1
    except ValueError:
        has_ip = 0

    uses_shortener = int(
        any(domain in hostname for domain in
            ["bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd"])
    )

    return [
        len(url),
        len(hostname),
        len(path),
        url.count("."),
        url.count("-"),
        url.count("@"),
        url.count("?"),
        url.count("="),
        url.count("/"),
        int(scheme == "https"),
        has_ip,
        sum(word in url_lower for word in suspicious_words),
        uses_shortener
    ]

feature_names = [
    "url_length", "hostname_length", "path_length",
    "dot_count", "hyphen_count", "at_count",
    "question_count", "equals_count", "slash_count",
    "uses_https", "has_ip", "suspicious_word_count",
    "uses_shortener"
]

X = pd.DataFrame(
    [extract_features(url) for url in df["url"]],
    columns=feature_names
)

y = (df["type"].str.lower() != "benign").astype(int)

print("Features created successfully!")
print("Feature shape:", X.shape)
print("Labels:")
print(y.value_counts())

ValueError: Invalid IPv6 URL